# Matrix Rank - Exercises

12 exercises covering [notes.md](notes.md) and [theory.ipynb](theory.ipynb).

**Format**:
- Problem: mathematical target plus AI framing where useful
- Your Solution: scaffold cell for your work
- Solution: one full worked answer with checks

| Level | Meaning | Exercises |
|---|---|---|
| * | Core rank mechanics | 1-4 |
| ** | Spectral and low-rank structure | 5-8 |
| *** | AI bottlenecks, LoRA, and numerical rank | 9-12 |

| Topic | Exercises |
|---|---|
| Pivots, null space, rank-nullity | 1, 2, 3 |
| Rank inequalities and structure | 4, 5 |
| Factorization and approximation | 6, 7, 8 |
| AI low-rank applications | 9, 10, 11, 12 |

In [ ]:
import numpy as np

np.set_printoptions(precision=6, suppress=True)


def header(title):
    print("\n" + "=" * len(title))
    print(title)
    print("=" * len(title))


def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    if not ok:
        print('expected:', expected)
        print('got     :', got)
    return ok


def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} - {name}")
    return cond

---

## Exercise 1: Rank from Pivots *

Use row reduction to determine the rank of a matrix. This is the algebraic starting point for the chapter.

**Task**:
- Implement a small `rref(M)` routine
- Return the reduced matrix and pivot columns
- Use it to compute the rank of a sample matrix

In [ ]:
# Exercise 1: Your Solution

def rref(M, tol=1e-12):
    M = np.array(M, dtype=float).copy()
    # YOUR CODE HERE
    pass


A = np.array([[1.0, 2.0, 3.0], [2.0, 4.0, 6.0], [1.0, 3.0, 4.0]])
R, pivots = rref(A)
print(R)
print('pivots:', pivots)

In [ ]:
# Exercise 1: Solution

def rref(M, tol=1e-12):
    M = np.array(M, dtype=float).copy()
    rows, cols = M.shape
    pivots = []
    r = 0

    for c in range(cols):
        if r >= rows:
            break
        pivot = r + np.argmax(np.abs(M[r:, c]))
        if abs(M[pivot, c]) < tol:
            continue
        if pivot != r:
            M[[r, pivot]] = M[[pivot, r]]
        M[r] /= M[r, c]
        for i in range(rows):
            if i != r and abs(M[i, c]) > tol:
                M[i] -= M[i, c] * M[r]
        M[np.abs(M) < tol] = 0.0
        pivots.append(c)
        r += 1
    return M, pivots


R, pivots = rref(A)

header('Exercise 1 checks')
print('RREF(A) =\n', R)
print('pivots =', pivots)
check_true('rank is 2', len(pivots) == 2)
print('\nTakeaway: in exact algebra, rank is the number of pivot columns.')

---

## Exercise 2: Null Space and Rank-Nullity *

Rank tells you how many directions survive. Nullity tells you how many are killed.

**Task**:
- Compute a basis for the null space using SVD
- Verify rank-nullity for a rectangular matrix

In [ ]:
# Exercise 2: Your Solution

def nullspace(A, tol=1e-10):
    A = np.array(A, dtype=float)
    # YOUR CODE HERE
    pass


B = np.array([[1.0, 0.0, 2.0, 1.0], [0.0, 1.0, 1.0, 2.0], [1.0, 1.0, 3.0, 3.0]])
N, rank_B = nullspace(B)
print('rank(B) =', rank_B)
print('nullspace basis =\n', N)

In [ ]:
# Exercise 2: Solution

def nullspace(A, tol=1e-10):
    U, s, Vt = np.linalg.svd(A)
    rank = np.sum(s > tol)
    return Vt[rank:].T, rank


N, rank_B = nullspace(B)
nullity_B = B.shape[1] - rank_B

header('Exercise 2 checks')
print('rank(B) =', rank_B)
print('nullity(B) =', nullity_B)
print('nullspace basis =\n', N)
check_true('rank-nullity holds', rank_B + nullity_B == B.shape[1])
check_close('B @ N = 0', B @ N, np.zeros((B.shape[0], N.shape[1])))
print('\nTakeaway: rank counts surviving directions, nullity counts annihilated directions.')

---

## Exercise 3: Row Space and Column Space Dimensions *

The theorem `row rank = column rank` says two very different-looking spaces have the same dimension.

**Task**:
- Find a basis for the row space from the non-zero rows of RREF
- Find a basis for the column space from the pivot columns of the original matrix
- Verify that the dimensions agree

In [ ]:
# Exercise 3: Your Solution

C = np.array([[1.0, 2.0, 0.0, -1.0], [2.0, 4.0, 1.0, 1.0], [-1.0, -2.0, 1.0, 3.0], [0.0, 0.0, 2.0, 4.0]])
R, pivots = rref(C)

# YOUR CODE HERE
row_basis = None
col_basis = None
print(row_basis)
print(col_basis)

In [ ]:
# Exercise 3: Solution

row_basis = R[~np.all(np.isclose(R, 0.0), axis=1)]
col_basis = C[:, pivots]

header('Exercise 3 checks')
print('pivot columns =', pivots)
print('\nrow-space basis =\n', row_basis)
print('\ncolumn-space basis =\n', col_basis)
check_true('row-space dimension equals column-space dimension', row_basis.shape[0] == col_basis.shape[1])
print('\nTakeaway: row rank and column rank coincide even though the spaces live in different ambient dimensions.')

---

## Exercise 4: Rank Inequalities *

Rank obeys useful upper and lower bounds under addition and multiplication.

**Task**:
- Build small matrices showing that product rank can collapse dramatically
- Verify the inequalities numerically

In [ ]:
# Exercise 4: Your Solution

A = np.array([[1.0, 0.0], [0.0, 0.0]])
B = np.array([[0.0, 0.0], [0.0, 1.0]])
C = np.array([[1.0, 0.0], [0.0, 0.0]])
D = np.array([[0.0, 1.0], [0.0, 0.0]])

# YOUR CODE HERE
print('fill in rank computations')

In [ ]:
# Exercise 4: Solution

rank_A = np.linalg.matrix_rank(A)
rank_B = np.linalg.matrix_rank(B)
rank_AB = np.linalg.matrix_rank(A @ B)
rank_C = np.linalg.matrix_rank(C)
rank_D = np.linalg.matrix_rank(D)
rank_C_plus_D = np.linalg.matrix_rank(C + D)

header('Exercise 4 checks')
print('rank(A), rank(B), rank(AB) =', rank_A, rank_B, rank_AB)
print('rank(C), rank(D), rank(C + D) =', rank_C, rank_D, rank_C_plus_D)
check_true('product rank can drop to zero', rank_AB == 0)
check_true('subadditivity holds', rank_C_plus_D <= rank_C + rank_D)
print('\nTakeaway: multiplication never creates new independent directions, and addition is only subadditive.')

---

## Exercise 5: Numerical Rank and Thresholds **

Exact rank is discrete. Numerical rank depends on what you treat as negligible.

**Task**:
- Build a diagonal matrix with singular values `(10, 1, 1e-2, 1e-6)`
- Study how the detected rank changes as the threshold changes

In [ ]:
# Exercise 5: Your Solution

S = np.diag([10.0, 1.0, 1e-2, 1e-6])
thresholds = [1e-1, 1e-3, 1e-5, 1e-7]

# YOUR CODE HERE
for eps in thresholds:
    pass

In [ ]:
# Exercise 5: Solution

singular_values = np.linalg.svd(S, compute_uv=False)

header('Exercise 5 checks')
print('singular values =', singular_values)
for eps in thresholds:
    numerical_rank = np.sum(singular_values > eps)
    print(f'threshold {eps:>7.1e} -> numerical rank {numerical_rank}')
check_true('exact rank is 4', np.linalg.matrix_rank(S) == 4)
print('\nTakeaway: numerical rank is threshold-sensitive, which is exactly why SVD-based diagnostics are needed in practice.')

---

## Exercise 6: Rank Factorization **

A rank-`r` matrix factors through an `r`-dimensional latent space.

**Task**:
- Construct a rank factorization `A = B @ C` using the SVD
- Verify that the inner dimension equals the rank

In [ ]:
# Exercise 6: Your Solution

A = np.array([[2.0, 4.0, 6.0], [1.0, 2.0, 3.0], [0.0, 1.0, 1.5]])

# YOUR CODE HERE
B = None
C = None
print(B)
print(C)

In [ ]:
# Exercise 6: Solution

U, s, Vt = np.linalg.svd(A, full_matrices=False)
r = np.linalg.matrix_rank(A)
B = U[:, :r] @ np.diag(s[:r])
C = Vt[:r, :]

header('Exercise 6 checks')
print('rank(A) =', r)
print('B shape =', B.shape)
print('C shape =', C.shape)
check_close('A reconstructed from factors', B @ C, A)
check_true('inner dimension equals rank', B.shape[1] == r and C.shape[0] == r)
print('\nTakeaway: rank factorization makes the latent bottleneck explicit.')

---

## Exercise 7: Best Rank-1 Approximation **

Use truncated SVD to compute the best rank-1 approximation and measure how much of the matrix energy it captures.

In [ ]:
# Exercise 7: Your Solution

A = np.array([[3.0, 2.0, 2.0], [2.0, 3.0, -2.0]])

# YOUR CODE HERE
A1 = None
fro_error = None
variance_fraction = None
print(A1)
print(fro_error, variance_fraction)

In [ ]:
# Exercise 7: Solution

U, s, Vt = np.linalg.svd(A, full_matrices=False)
A1 = s[0] * np.outer(U[:, 0], Vt[0, :])
fro_error = np.linalg.norm(A - A1, ord='fro')
variance_fraction = (s[0] ** 2) / np.sum(s**2)

header('Exercise 7 checks')
print('singular values =', s)
print('\nA1 =\n', A1)
print('\nFrobenius error =', fro_error)
print('variance fraction =', variance_fraction)
check_true('Eckart-Young Frobenius error matches discarded singular value', np.isclose(fro_error, s[1]))
print('\nTakeaway: low-rank approximation quality is governed by singular value decay, not by matrix size alone.')

---

## Exercise 8: Stable Rank vs Exact Rank **

Two matrices can have the same exact rank but very different effective structure.

**Task**:
- Compute exact rank, stable rank, and condition number for several examples

In [ ]:
# Exercise 8: Your Solution

def stable_rank(A):
    # YOUR CODE HERE
    pass


W1 = np.diag([10.0, 5.0, 1.0, 0.1, 0.01])
W2 = np.eye(5)
u = np.array([[1.0], [2.0], [3.0], [4.0], [5.0]])
v = np.array([[2.0, -1.0, 0.5, 3.0, -2.0]])
W3 = u @ v

for name, W in [('W1', W1), ('W2', W2), ('W3', W3)]:
    print(name, stable_rank(W))

In [ ]:
# Exercise 8: Solution

def stable_rank(A):
    s = np.linalg.svd(A, compute_uv=False)
    return np.sum(s**2) / (s[0]**2)


header('Exercise 8 checks')
for name, W in [('W1', W1), ('W2', W2), ('W3', W3)]:
    rank = np.linalg.matrix_rank(W)
    sr = stable_rank(W)
    cond = np.linalg.cond(W) if rank == min(W.shape) else np.inf
    print(f'{name}: rank = {rank}, stable rank = {sr:.6f}, cond = {cond}')
check_true('W3 is rank 1', np.linalg.matrix_rank(W3) == 1)
check_true('stable rank never exceeds exact rank', stable_rank(W1) <= np.linalg.matrix_rank(W1))
print('\nTakeaway: stable rank measures how spread the spectrum is, not just how many singular values are non-zero.')

---

## Exercise 9: LoRA Parameter Budget ***

LoRA replaces a dense update with a rank-constrained one. The rank is the budget knob.

**Task**:
- For a `4096 x 4096` matrix, compute LoRA parameter counts for several ranks
- Express each as a fraction of full fine-tuning

In [ ]:
# Exercise 9: Your Solution

m = n = 4096
ranks = [1, 4, 8, 16, 32, 64]

# YOUR CODE HERE
for r in ranks:
    pass

In [ ]:
# Exercise 9: Solution

full_params = m * n

header('Exercise 9 checks')
for r in ranks:
    lora_params = r * (m + n)
    ratio = lora_params / full_params
    print(f'r = {r:>2}: params = {lora_params:>8}, fraction = {ratio:.6f}')
check_true('rank-8 LoRA is dramatically smaller than full fine-tuning', (8 * (m + n)) < full_params / 100)
print('\nTakeaway: rank is not abstract here; it is the adapter memory and compute budget.')

---

## Exercise 10: Attention Rank Ceiling ***

Attention score matrices look large, but their rank is constrained by the projection width `d_k`.

**Task**:
- Construct `Q` and `K` with shape `(n, d_k)`
- Form `S = Q @ K.T`
- Verify `rank(S) <= d_k`

In [ ]:
# Exercise 10: Your Solution

n = 4
d_k = 2
Q = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0], [2.0, -1.0]])
K = np.array([[1.0, 1.0], [1.0, 0.0], [0.0, 1.0], [2.0, 1.0]])

# YOUR CODE HERE
S = None
print(S)

In [ ]:
# Exercise 10: Solution

S = Q @ K.T
rank_S = np.linalg.matrix_rank(S)

header('Exercise 10 checks')
print('S =\n', S)
print('rank(S) =', rank_S)
check_true('score matrix rank is bounded by d_k', rank_S <= d_k)
print('\nTakeaway: even a large attention score matrix is generated through a low-rank projection bottleneck.')

---

## Exercise 11: Rank-Deficient Least Squares ***

If a matrix is rank-deficient, least squares still makes sense, but uniqueness now depends on the minimum-norm criterion.

**Task**:
- Solve a rank-deficient least-squares problem using the pseudo-inverse
- Verify that the residual is orthogonal to the column space

In [ ]:
# Exercise 11: Your Solution

A = np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]])
b = np.array([1.0, 2.1, 2.9])

# YOUR CODE HERE
x_star = None
residual = None
print(x_star)
print(residual)

In [ ]:
# Exercise 11: Solution

x_star = np.linalg.pinv(A) @ b
residual = b - A @ x_star
projection_test = A.T @ residual

header('Exercise 11 checks')
print('x_star =', x_star)
print('residual =', residual)
print('A^T residual =', projection_test)
check_close('residual orthogonal to column space', projection_test, np.zeros(A.shape[1]))
print('\nTakeaway: rank deficiency changes uniqueness, but the pseudo-inverse still gives the minimum-norm least-squares solution.')

---

## Exercise 12: Latent Bottleneck Rank Ceiling ***

A down-projection followed by an up-projection cannot have rank larger than the latent width.

**Task**:
- Build `W_down` with shape `(d, r)` and `W_up` with shape `(r, d)`
- Form the effective map `W_eff = W_down @ W_up`
- Verify that `rank(W_eff) <= r`

This is the linear-algebra core of latent bottleneck architectures.

In [ ]:
# Exercise 12: Your Solution

d = 16
r = 4
np.random.seed(0)
W_down = np.random.randn(d, r)
W_up = np.random.randn(r, d)

# YOUR CODE HERE
W_eff = None
print(W_eff.shape)

In [ ]:
# Exercise 12: Solution

W_eff = W_down @ W_up
rank_eff = np.linalg.matrix_rank(W_eff)

header('Exercise 12 checks')
print('W_eff shape =', W_eff.shape)
print('rank(W_eff) =', rank_eff)
print('latent width =', r)
check_true('effective rank is bounded by latent width', rank_eff <= r)
print('\nTakeaway: low-rank latent bottlenecks are exact rank constraints, not just approximate intuitions.')

---

## What to Review After Finishing

- Can you compute rank from pivots, from null space dimension, and from singular values?
- Can you explain why row rank and column rank are the same number even though the spaces differ?
- Can you use SVD to reason about numerical rank and low-rank approximation?
- Can you explain LoRA and attention bottlenecks directly in rank language?
- Do you know when exact rank is the wrong practical concept and numerical rank is the right one?

References used in the chapter and notebook design:
- [notes.md](notes.md)
- [theory.ipynb](theory.ipynb)
- [MIT 18.06 Linear Algebra](https://web.mit.edu/18.06/www/)
- [Stanford EE263](https://stanford.edu/class/ee263/)
- [LoRA](https://arxiv.org/abs/2106.09685)
- [DeepSeek-V2 Technical Report](https://arxiv.org/abs/2405.04434)
- [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437)
- [Aghajanyan et al. (2020)](https://arxiv.org/abs/2012.13255)
- [Halko, Martinsson, and Tropp (2011)](https://arxiv.org/abs/0909.4061)
- [Gavish and Donoho (2014)](https://arxiv.org/abs/1305.5870)